In [15]:
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from scipy.interpolate import griddata

In [16]:
df = pd.read_csv('results.csv')
print(df.shape)
df.head()

(754, 4)


,instance_no,L2_value,drop_value,loss
0,0,0.000162,0.348326,0.636713
1,1,0.001153,0.137565,0.640184
2,2,0.001867,0.408430,0.645374
3,3,0.000098,0.002429,0.637338
4,4,0.000004,0.136765,0.634278


In [17]:
df

,instance_no,L2_value,drop_value,loss
0,0,0.000162,0.348326,0.636713
1,1,0.001153,0.137565,0.640184
2,2,0.001867,0.408430,0.645374
3,3,0.000098,0.002429,0.637338
4,4,0.000004,0.136765,0.634278
...,...,...,...,...
749,749,0.000002,0.400326,0.629485
750,750,0.000014,0.461910,0.633757
751,751,0.000067,0.214278,0.630330
752,752,0.000080,0.086990,0.632378


In [18]:
# Flipping the loss and normalizing the y axis to get AUC
df['auc'] = (-df['loss'] + df['loss'].max()) * (1 / (df['loss'].max() * df.shape[0]))
print(df.shape)
df.head()

(754, 5)


,instance_no,L2_value,drop_value,loss,auc
0,0,0.000162,0.348326,0.636713,0.000032
1,1,0.001153,0.137565,0.640184,0.000025
2,2,0.001867,0.408430,0.645374,0.000015
3,3,0.000098,0.002429,0.637338,0.000031
4,4,0.000004,0.136765,0.634278,0.000037


In [31]:
def plot_3d_scatter_surface(df, x_variable, y_variables, output_variable, surface=True, highlight=False):
    name = ['L2', 'Dropout']
    
    for i, y_var in enumerate(y_variables):
        print(f'{name[i]}: Min value = {df[y_var].min()}, Max value = {df[y_var].max()}, Min Loss = {df[output_variable].min()}')
        # Find the index of the lowest output variable value if highlight is True
        lowest_index = df[output_variable].idxmin() if highlight else None
        
        # Create a scatter plot
        scatter = go.Scatter3d(
            x=df[x_variable],
            y=df[y_var],
            z=df[output_variable],
            mode='markers',
            marker=dict(
                size=[15 if i == lowest_index else 5 for i in range(len(df))],  # Highlighted point is bigger,
                color=['red' if i == lowest_index else 'grey' for i in range(len(df))] if highlight else 'grey',  # Highlight the lowest value if highlight is True
                colorscale='Viridis',  # Choose a color scale
                opacity=0.8,
            ),
            name=output_variable
        )

        # Create surface data
        x_data = df[x_variable]
        y_data = df[y_var]
        z_data = df[output_variable]

        x_range = np.linspace(min(x_data), max(x_data), 100)
        y_range = np.linspace(min(y_data), max(y_data), 100)
        X, Y = np.meshgrid(x_range, y_range)
        Z = griddata((x_data, y_data), z_data, (X, Y), method='cubic')

        if surface:
            # Create a 3D surface
            surface = go.Surface(
                x=X,
                y=Y,
                z=Z,
                colorscale='Jet',  # Choose a color scale
                opacity=0.6
            )

            # Create the figure
            fig = go.Figure(data=[scatter, surface])

        else:
            fig = go.Figure(data=[scatter])

        # Update layout
        fig.update_layout(
            scene=dict(
                xaxis_title=x_variable,
                yaxis_title=y_var,
                zaxis_title=output_variable
            ),
            width=800,  # adjust width
            height=600,  # adjust height
            title=f'Mean Loss for different values of {name[i]} regularization'
        )

        # Show the plot
        fig.show()

In [33]:
plot_3d_scatter_surface(df, 'instance_no', ['L2_value', 'drop_value'], 'loss', surface=False, highlight=True)

L2: Min value = 1.05e-06, Max value = 0.004160265, Min Loss = 0.619165382


Dropout: Min value = 0.001647456, Max value = 0.499534249, Min Loss = 0.619165382


In [21]:
def plot_2d_scatter(df, x_column, output_column, highlight=False):

    # Find the index of the lowest output variable value if highlight is True
    lowest_index = df[output_column].idxmin() if highlight else None
    
    # Create a scatter plot
    scatter = go.Scatter(
        x=df[x_column],
        y=df[output_column],
        mode='lines+markers',
    
        marker=dict(
            size=[15 if i == lowest_index else 5 for i in range(len(df))],  # Highlighted point is bigger
            color=['red' if i == lowest_index else 'blue' for i in range(len(df))] if highlight else 'blue',  # Highlight the lowest value if highlight is True
            opacity=0.8,
        ),
        name=output_column
    )

    # Create the figure
    fig = go.Figure(data=[scatter])

    # Update layout
    fig.update_layout(
        xaxis_title=x_column,
        yaxis_title=output_column,
        # yaxis=dict(tickformat='e'),
        width=800,  # adjust width
        height=600,  # adjust height
        title=f'Loss as a function of iterations'
    )

    # Show the plot
    fig.show()

In [22]:
plot_2d_scatter(df, 'instance_no', 'loss', highlight=True)

In [23]:
auc = np.trapz(df['auc'], x=df['instance_no'])
print(f'Area under the curve (AUC): {auc:.2f}')

plot_2d_scatter(df, 'instance_no', 'auc', highlight=False)

Area under the curve (AUC): 0.02
